In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
from utils.transformations import reusable

### DimUser###


### Assigning system path###

In [0]:
print(os.getcwd())

In [0]:
project_path=os.path.abspath(os.path.join(os.getcwd(),'..','..'))
if project_path not in sys.path:
    sys.path.append(project_path)
import importlib
import utils.transformations as transformations
importlib.reload(transformations)


In [0]:
print(project_path)

###Autoloader###

###DimUser###

In [0]:
df_user=spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimUser/schema")\
            .load("abfss://bronze@adeprojectstorage.dfs.core.windows.net/DimUser")
df_user=df_user.withColumn("user_name",upper(col("user_name"))) 
df_user=reusable().dropColumns(df_user,['_rescued_data'])
df_user.dropDuplicates(['user_id'])           
df_user.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimUser/checkpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimUser/data")\
            .toTable("spotify_cata.silver.DimUser")           

In [0]:
df_user_output=spark.read.format("delta").load("abfss://silver@adeprojectstorage.dfs.core.windows.net/DimArtist/data") 
display(df_user_output)

###DimArtist###

In [0]:
df_artist=spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimArtist/schema")\
            .load("abfss://bronze@adeprojectstorage.dfs.core.windows.net/DimArtist")
df_artist=df_artist.withColumn("artist_name",upper(col("artist_name"))) 
df_artist=reusable().dropColumns(df_artist,['_rescued_data'])
df_artist.dropDuplicates(['artist_id'])              
df_artist.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimArtist/checkpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimArtist/data")\
            .toTable("spotify_cata.silver.DimArtist")

In [0]:
df_artist_output=spark.read.format("delta").load("abfss://silver@adeprojectstorage.dfs.core.windows.net/DimArtist/data") 
display(df_artist_output)

###DimTrack###

In [0]:
df_track=spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimTrack/schema")\
            .load("abfss://bronze@adeprojectstorage.dfs.core.windows.net/DimTrack")
df_track=df_track.withColumn("durationFlag", 
                                when(col('duration_sec')<150, "low")
                                .when(col('duration_sec')<300, "medium")
                                .otherwise("high") )
df_track=df_track.withColumn("track_name",regexp_replace(col("track_name"),"-"," "))         
df_track=reusable().dropColumns(df_track,['_rescued_data'])               
df_track.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimTrack/checkpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimTrack/data")\
            .toTable("spotify_cata.silver.DimTrack")

In [0]:
df_track_output=spark.read.format("delta").load("abfss://silver@adeprojectstorage.dfs.core.windows.net/DimTrack/data") 
display(df_track_output)

###DimDate###

In [0]:
df_date=spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimDate/schema")\
            .load("abfss://bronze@adeprojectstorage.dfs.core.windows.net/DimDate")
df_date=reusable().dropColumns(df_date,['_rescued_data'])            
df_date.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimDate/checkpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@adeprojectstorage.dfs.core.windows.net/DimDate/data")\
            .toTable("spotify_cata.silver.DimDate")

In [0]:
df_date_output=spark.read.format("delta").load("abfss://silver@adeprojectstorage.dfs.core.windows.net/DimDate/data") 
display(df_date_output)

###FactStream###

In [0]:
df_fact=spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/FactStream/schema")\
            .load("abfss://bronze@adeprojectstorage.dfs.core.windows.net/FactStream")
df_fact=reusable().dropColumns(df_fact,['_rescued_data'])                     
df_fact.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@adeprojectstorage.dfs.core.windows.net/FactStream/checkpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@adeprojectstorage.dfs.core.windows.net/FactStream/data")\
            .toTable("spotify_cata.silver.FactStream")

In [0]:
df_fact_output=spark.read.format("delta").load("abfss://silver@adeprojectstorage.dfs.core.windows.net/FactStream/data") 
display(df_fact_output)